## 1. 라이브러리 로드

In [1]:
# 라이브러리 호출
import pandas as pd
import numpy as np
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams
import platform
import ast
from collections import Counter
import json

warnings.filterwarnings('ignore')

# 한글 폰트 설정
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # Mac
    plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

# 컬럼 너비 제한 해제
pd.set_option('display.max_colwidth', None)

---
## 2. 데이터 로드

In [2]:
df_platinum_Match = pd.read_csv('./tft_game_dataset/TFT_Platinum_MatchData.csv')
df_diamond_Match = pd.read_csv('./tft_game_dataset/TFT_Diamond_MatchData.csv')
df_master_Match = pd.read_csv('./tft_game_dataset/TFT_Master_MatchData.csv')
df_grand_master_Match = pd.read_csv('./tft_game_dataset/TFT_GrandMaster_MatchData.csv')
df_challenger_Match = pd.read_csv('./tft_game_dataset/TFT_Challenger_MatchData.csv')

df_champion_info = pd.read_csv('./tft_game_dataset/TFT_Champion_CurrentVersion.csv')
df_items_info = pd.read_csv('./tft_game_dataset/TFT_Item_CurrentVersion.csv')


In [3]:
# 매치관련 데이터가 담긴 딕셔너리 정의
match_data = {
    'platinum': df_platinum_Match,
    'diamond': df_diamond_Match,
    'master': df_master_Match,
    'grand_master': df_grand_master_Match,
    'challenger': df_challenger_Match,
}

# 각 티어별 테이블에 'Tier'정보가 담긴 컬럼 추가
for name, df in match_data.items():
    df['user_tier'] = name


# 모든 티어의 경기데이터가 담긴 데이터프레임 제작
# ignore_index=True: 데이터를 병합한 뒤, 새로운 인덱스 부여
df_all_match = pd.concat(match_data.values(), ignore_index=True)

df_all_match.head(3)

,gameId,gameDuration,level,lastRound,Ranked,ingameDuration,combination,champion,user_tier
0,KR_4291707834,1963.905273,6,27,5,1390.165771,"{'Cybernetic': 1, 'Demolitionist': 1, 'Infiltrator': 1, 'Rebel': 1, 'Set3_Brawler': 1, 'Set3_Celestial': 1, 'Set3_Void': 1, 'Sniper': 1}","{'Ziggs': {'items': [7], 'star': 1}, 'Ashe': {'items': [9], 'star': 1}, 'ChoGath': {'items': [6], 'star': 1}, 'Ekko': {'items': [1], 'star': 1}}",platinum
1,KR_4291707834,1963.905273,8,37,3,1891.282715,"{'Blaster': 1, 'Chrono': 1, 'Cybernetic': 4, 'Demolitionist': 1, 'Rebel': 1, 'Set3_Blademaster': 2, 'Set3_Brawler': 1, 'Set3_Sorcerer': 1, 'Set3_Void': 1, 'Valkyrie': 1, 'Vanguard': 2}","{'Ziggs': {'items': [24], 'star': 3}, 'Fiora': {'items': [37], 'star': 2}, 'Leona': {'items': [36, 24], 'star': 2}, 'Lucian': {'items': [], 'star': 2}, 'Vi': {'items': [5], 'star': 2}, 'Kayle': {'items': [], 'star': 2}, 'WuKong': {'items': [3, 67], 'star': 2}, 'VelKoz': {'items': [4], 'star': 2}}",platinum
2,KR_4291707834,1963.905273,6,25,7,1279.461060,"{'Blaster': 1, 'Cybernetic': 1, 'DarkStar': 2, 'Infiltrator': 1, 'Mercenary': 1, 'Set3_Blademaster': 1, 'Set3_Mystic': 1, 'Valkyrie': 1}","{'Fiora': {'items': [1], 'star': 1}, 'Shaco': {'items': [6], 'star': 1}, 'Karma': {'items': [4], 'star': 1}, 'MissFortune': {'items': [3], 'star': 1}}",platinum


---
## 3. 데이터 전처리
### 3-1. 중복행 제거

In [4]:
# 중복행 제거
duplicates = df_all_match[df_all_match.duplicated(keep=False)]

print(f"중복 제거 전: {len(df_all_match)}")
print(f"중복된 행 개수: {len(duplicates)}")

# 결과 확인
df_all_match = df_all_match.drop_duplicates()
print(f"\n중복 제거 후: {len(df_all_match)}")

중복 제거 전: 399998
중복된 행 개수: 80

중복 제거 후: 399930


### 3-2. 컬럼명 소문자로 통일

In [5]:
# 전체 컬럼명 소문자 통일
df_all_match.columns = df_all_match.columns.str.lower()

print(df_all_match.columns)

Index(['gameid', 'gameduration', 'level', 'lastround', 'ranked',
       'ingameduration', 'combination', 'champion', 'user_tier'],
      dtype='str')


### 3-3. 시즌2 데이터 삭제

In [6]:
df_all_match_2 = df_all_match.copy()

# 시즌 2 고유 시너지 목록
season2_keys = {'Alchemist', 'Avatar', 'Berserker',
                'Crystal', 'Desert', 'Druid', 'Electric',
                'Light', 'Mage', 'Mountain', 'Mystic', 'Ocean',
                'Poison', 'Predator', 'Set2_Assassin', 'Set2_Blademaster',
                'Set2_Glacial', 'Set2_Ranger', 'Shadow', 'Soulbound',
                'Summoner', 'Warden', 'Wind', 'Woodland'}  # 시즌 2 시너지 입력


# 시즌 2 시너지가 하나라도 포함되면 시즌 2로 분류
df_all_match_2['season'] = df_all_match_2['combination'].apply(
    lambda x: 'season 2' if any(k in season2_keys for k in ast.literal_eval(x).keys())
    else 'season 3'
)

# season 2인 행의 gameId 추출
drop_game_ids_4 = df_all_match_2[df_all_match_2['season'] == 'season 2']['gameid'].unique()


# 해당 gameId를 가진 행의 인덱스 추출 후 삭제
drop_idx_4 = df_all_match_2[df_all_match_2['gameid'].isin(drop_game_ids_4)].index
df_all_match_2 = df_all_match_2.drop(index=drop_idx_4)

print(f"season 2인 데이터가 포함된 경기 데이터 삭제한 후: {df_all_match_2.shape}")

season 2인 데이터가 포함된 경기 데이터 삭제한 후: (396698, 10)


### 3-4. 시즌 3 시너지 더미 데이터 ”TemplateTrait” 삭제
- 시너지 데이터 중 해당 값인 `{'TemplateTrait': 1}`만 삭제

In [7]:
# TemplateTrait 키가 있는 행만 필터링 후 값 확인
df_all_match_2[df_all_match_2['combination'].apply(
    lambda x: 'TemplateTrait' in ast.literal_eval(x).keys() # TemplateTrait 키가 있는 행만 필터링
)]['combination'].apply(                                    # 필터링된 행 중에서 combination 컬럼만 가져옴
    lambda x: ast.literal_eval(x).get('TemplateTrait')      # TemplateTrait 키에 할당된 값을 가져옴
).value_counts()


combination
1    65988
Name: count, dtype: int64

In [8]:
# 분석 목적에 영향을 주지 않는 dummy 데이터로 판단하여 시너지 중 'TemplateTrait'만 삭제
df_all_match_2['combination'] = df_all_match_2['combination'].apply(
    lambda x: json.dumps(
        # key: value = 키-값의 쌍을 의미함
        {key: value for key, value in ast.literal_eval(x).items() if key != 'TemplateTrait'},
        ensure_ascii=False
    )
)

In [9]:
# TemplateTrait 키가 있는 행 수 확인 → (0,10)이면 완전히 삭제된 것
df_all_match_2[df_all_match_2['combination'].apply(
    lambda x: 'TemplateTrait' in ast.literal_eval(x).keys()
)].shape

(0, 10)

### 3-5. ranked=0인 데이터 삭제
- game_id 단위로 진행

In [10]:
# ranked==0인 행의 gameId 추출
drop_game_ids_1 = df_all_match_2[df_all_match_2['ranked']==0]['gameid'].unique()

# 해당 gameId를 가진 행의 인덱스 추출 후 삭제
drop_idx_1 = df_all_match_2[df_all_match_2['gameid'].isin(drop_game_ids_1)].index
df_all_match_2 = df_all_match_2.drop(index=drop_idx_1)

print(f"ranked = 0인 데이터가 포함된 경기 데이터 삭제한 후: {df_all_match_2.shape}")

ranked = 0인 데이터가 포함된 경기 데이터 삭제한 후: (396654, 10)


### 3-6. USER_ID 컬럼 제작
- `KR-USER-(인덱스+1)`의 규칙으로 부여

In [11]:
df_all_match_2['user_id'] = 'KR-USER-' + (df_all_match_2.index + 1).astype(str)

df_all_match_2['user_id']

0              KR-USER-1
1              KR-USER-2
2              KR-USER-3
3              KR-USER-4
4              KR-USER-5
               ...      
399993    KR-USER-399994
399994    KR-USER-399995
399995    KR-USER-399996
399996    KR-USER-399997
399997    KR-USER-399998
Name: user_id, Length: 396654, dtype: str

In [ ]:
display(df_all_match_2.head())
print(df_all_match_2.shape) # (396654, 11)

,gameid,gameduration,level,lastround,ranked,ingameduration,combination,champion,user_tier,season,user_id
0,KR_4291707834,1963.905273,6,27,5,1390.165771,"{""Cybernetic"": 1, ""Demolitionist"": 1, ""Infiltrator"": 1, ""Rebel"": 1, ""Set3_Brawler"": 1, ""Set3_Celestial"": 1, ""Set3_Void"": 1, ""Sniper"": 1}","{'Ziggs': {'items': [7], 'star': 1}, 'Ashe': {'items': [9], 'star': 1}, 'ChoGath': {'items': [6], 'star': 1}, 'Ekko': {'items': [1], 'star': 1}}",platinum,season 3,KR-USER-1
1,KR_4291707834,1963.905273,8,37,3,1891.282715,"{""Blaster"": 1, ""Chrono"": 1, ""Cybernetic"": 4, ""Demolitionist"": 1, ""Rebel"": 1, ""Set3_Blademaster"": 2, ""Set3_Brawler"": 1, ""Set3_Sorcerer"": 1, ""Set3_Void"": 1, ""Valkyrie"": 1, ""Vanguard"": 2}","{'Ziggs': {'items': [24], 'star': 3}, 'Fiora': {'items': [37], 'star': 2}, 'Leona': {'items': [36, 24], 'star': 2}, 'Lucian': {'items': [], 'star': 2}, 'Vi': {'items': [5], 'star': 2}, 'Kayle': {'items': [], 'star': 2}, 'WuKong': {'items': [3, 67], 'star': 2}, 'VelKoz': {'items': [4], 'star': 2}}",platinum,season 3,KR-USER-2
2,KR_4291707834,1963.905273,6,25,7,1279.461060,"{""Blaster"": 1, ""Cybernetic"": 1, ""DarkStar"": 2, ""Infiltrator"": 1, ""Mercenary"": 1, ""Set3_Blademaster"": 1, ""Set3_Mystic"": 1, ""Valkyrie"": 1}","{'Fiora': {'items': [1], 'star': 1}, 'Shaco': {'items': [6], 'star': 1}, 'Karma': {'items': [4], 'star': 1}, 'MissFortune': {'items': [3], 'star': 1}}",platinum,season 3,KR-USER-3
3,KR_4291707834,1963.905273,7,38,2,1955.608521,"{""DarkStar"": 1, ""Protector"": 2, ""Set3_Blademaster"": 1, ""Set3_Celestial"": 5, ""Set3_Mystic"": 1, ""Sniper"": 1, ""StarGuardian"": 2, ""Vanguard"": 2}","{'Poppy': {'items': [], 'star': 2}, 'Xayah': {'items': [19, 23, 19], 'star': 3}, 'Rakan': {'items': [], 'star': 2}, 'XinZhao': {'items': [16], 'star': 2}, 'Mordekaiser': {'items': [35, 67, 33], 'star': 3}, 'Ashe': {'items': [], 'star': 2}, 'Soraka': {'items': [68, 47], 'star': 2}}",platinum,season 3,KR-USER-4
4,KR_4291707834,1963.905273,8,38,1,1955.608521,"{""Blaster"": 1, ""Chrono"": 5, ""DarkStar"": 3, ""Protector"": 1, ""Set3_Blademaster"": 1, ""Set3_Brawler"": 1, ""Set3_Sorcerer"": 2, ""Sniper"": 2}","{'TwistedFate': {'items': [36, 27], 'star': 3}, 'Caitlyn': {'items': [49, 29], 'star': 2}, 'JarvanIV': {'items': [56], 'star': 2}, 'Blitzcrank': {'items': [15], 'star': 2}, 'Shen': {'items': [77, 6], 'star': 2}, 'Ezreal': {'items': [16], 'star': 2}, 'Lux': {'items': [], 'star': 2}, 'Jhin': {'items': [], 'star': 2}}",platinum,season 3,KR-USER-5


(396654, 11)


### 3-7. combination과 champion 정보가 모두 누락된 행 삭제
- 두 컬럼이 모두 빈 딕셔너리 형태의 값을 갖고 있는 행을 삭제함

In [ ]:
# 두 컬럼 모두 빈 딕셔너리인 행 삭제
drop_idx_3 = df_all_match_2[
    df_all_match_2['combination'].apply(lambda x: ast.literal_eval(x) == {}) &
    df_all_match_2['champion'].apply(lambda x: ast.literal_eval(x) == {})
].index

df_all_match_2 = df_all_match_2.drop(index=drop_idx_3)

print(df_all_match_2.shape) # (396391, 11)

(396391, 11)


### 3-8. combination 정보는 있지만 champion 정보는 없는 행 처리
- 0,1로 구성된 `flag_1` 파생컬럼을 만들어서 combination만 활용하고, 사후검정용으로 활용할 예정

In [14]:
# flag_1 컬럼 생성
# combination은 값이 있고 champion은 빈 딕셔너리인 경우 True -> 1

df_all_match_2['flag_1'] = (
    df_all_match_2['combination'].apply(lambda x: ast.literal_eval(x) != {}) &
    df_all_match_2['champion'].apply(lambda x: ast.literal_eval(x) == {})
).astype(int)

# 결과 확인
df_all_match_2['flag_1'].value_counts()

flag_1
0    396356
1        35
Name: count, dtype: int64

### 3-9. champion 정보는 있지만 combination 정보는 없는 행 처리
- champion 정보를 기반으로 combination 정보를 복원함
- 정보를 복원한 행에 대해서는 0,1로 구성된 `flag_2` 파생컬럼을 만들어 확인할 예정

In [15]:
# flag_2 컬럼 생성
# combination은 값이 있고 champion은 빈 딕셔너리인 경우 True -> 1

df_all_match_2['flag_2'] = (
    df_all_match_2['combination'].apply(lambda x: ast.literal_eval(x) == {}) &
    df_all_match_2['champion'].apply(lambda x: ast.literal_eval(x) != {})
).astype(int)

# 결과 확인
df_all_match_2['flag_2'].value_counts()

flag_2
0    396274
1       117
Name: count, dtype: int64

---
## 4. 티어가 섞인 경기에 대한 처리

In [16]:
# 티어가 섞인 gameId 추출
mixed_gameids = df_all_match_2.groupby('gameid')['user_tier'].nunique() # gameid별로 tier의 고유값 개수를 계산
mixed_gameids = mixed_gameids[mixed_gameids > 1].index # 고유값이 2 이상인 gameId만 추출

print(len((mixed_gameids)))

# 해당 gameId의 티어 구성 확인
# mixed_gameids에 포함된 gameId를 가진 행만 추출 -> gameid별로 어떤 티어가 몇 명씩 있는지 카운트
df_all_match_2[df_all_match_2['gameid'].isin(mixed_gameids)].groupby('gameid')['user_tier'].value_counts()

19


gameid         user_tier   
KR_4263820118  platinum        8
               master          8
KR_4313697214  master          8
               platinum        8
KR_4320079059  platinum        8
               diamond         8
KR_4344513862  master          8
               diamond         8
KR_4347884427  diamond         8
               master          8
KR_4357966612  grand_master    8
               platinum        8
KR_4358922415  master          8
               diamond         8
KR_4361594426  master          8
               diamond         8
KR_4361773461  diamond         8
               grand_master    8
KR_4362846604  master          8
               diamond         8
KR_4364453473  grand_master    8
               diamond         8
KR_4365284161  master          8
               diamond         8
KR_4378896137  diamond         8
               platinum        8
KR_4381231118  diamond         8
               platinum        8
KR_4387025966  diamond         8
               

In [17]:
# 티어 순서 정의 (숫자가 높을수록 높은 티어)
tier_order = {
    'platinum': 1,
    'diamond': 2,
    'master': 3,
    'grand_master': 4,
    'challenger': 5
}

In [18]:
# 기본값 normal로 설정
df_all_match_2['mix_flag'] = 'normal'

# gameId별 최고/최저 티어 추출
highest_tier = {}
lowest_tier = {}

for gameid, group in df_all_match_2[df_all_match_2['gameid'].isin(mixed_gameids)].groupby('gameid')['user_tier']:
    tier_numbers = [tier_order[t] for t in group]
    
    max_tier_number = max(tier_numbers)
    min_tier_number = min(tier_numbers)
    
    for t in tier_order:
        if tier_order[t] == max_tier_number:
            highest_tier[gameid] = t
        if tier_order[t] == min_tier_number:
            lowest_tier[gameid] = t

# 믹스매치 행에 high/low 플래그 부여
for idx, row in df_all_match_2[df_all_match_2['gameid'].isin(mixed_gameids)].iterrows():
    if row['user_tier'] == highest_tier[row['gameid']]:
        df_all_match_2.loc[idx, 'mix_flag'] = 'high'
    else:
        df_all_match_2.loc[idx, 'mix_flag'] = 'low'

# 확인
df_all_match_2['mix_flag'].value_counts()

mix_flag
normal    396087
low          152
high         152
Name: count, dtype: int64

### 4-1.동일 게임 ID를 기준으로 여러 티어가 섞여있는 게임이라면, 상위 티어를 가진 유저의 정보를 보존함

In [21]:
df_all_match_5 = df_all_match_2[df_all_match_2['mix_flag'].isin(['normal', 'high'])]

print(df_all_match_5.shape)
print('='*50)
print(df_all_match_5.columns)
print('='*50)
print(df_all_match_5['mix_flag'].value_counts())

(396239, 14)
Index(['gameid', 'gameduration', 'level', 'lastround', 'ranked',
       'ingameduration', 'combination', 'champion', 'user_tier', 'season',
       'user_id', 'flag_1', 'flag_2', 'mix_flag'],
      dtype='str')
mix_flag
normal    396087
high         152
Name: count, dtype: int64


### 4-2.동일 게임 ID를 기준으로 여러 티어가 섞여있는 게임이라면, 하위 티어를 가진 유저의 정보를 보존함

In [22]:
df_all_match_6 = df_all_match_2[df_all_match_2['mix_flag'].isin(['normal', 'low'])]

print(df_all_match_6.shape)
print('='*50)
print(df_all_match_6.columns)
print('='*50)
print(df_all_match_6['mix_flag'].value_counts())

(396239, 14)
Index(['gameid', 'gameduration', 'level', 'lastround', 'ranked',
       'ingameduration', 'combination', 'champion', 'user_tier', 'season',
       'user_id', 'flag_1', 'flag_2', 'mix_flag'],
      dtype='str')
mix_flag
normal    396087
low          152
Name: count, dtype: int64


---
## 5.챔피언+아이템 정보를 활용하여 콤비네이션 컬럼 채우기

In [23]:
# 챔피언별 시너지 목록
champion_traits = {
    'Ahri': ['StarGuardian', 'Set3_Sorcerer'],
    'Annie': ['MechPilot', 'Set3_Sorcerer'],
    'Ashe': ['Set3_Celestial', 'Sniper'],
    'AurelionSol': ['Rebel', 'Starship'],
    'Blitzcrank': ['Chrono', 'Set3_Brawler'],
    'Caitlyn': ['Chrono', 'Sniper'],
    'ChoGath': ['Set3_Void', 'Set3_Brawler'],
    'Darius': ['SpacePirate', 'ManaReaver'],
    'Ekko': ['Cybernetic', 'Infiltrator'],
    'Ezreal': ['Chrono', 'Blaster'],
    'Fiora': ['Cybernetic', 'Set3_Blademaster'],
    'Fizz': ['MechPilot', 'Infiltrator'],
    'Gangplank': ['SpacePirate', 'Demolitionist', 'Mercenary'],
    'Graves': ['SpacePirate', 'Blaster'],
    'Irelia': ['Cybernetic', 'Set3_Blademaster', 'ManaReaver'],
    'JarvanIV': ['DarkStar', 'Protector'],
    'Jayce': ['SpacePirate', 'Vanguard'],
    'Jhin': ['DarkStar', 'Sniper'],
    'Jinx': ['Rebel', 'Blaster'],
    'KaiSa': ['Valkyrie', 'Infiltrator'],
    'Karma': ['DarkStar', 'Set3_Mystic'],
    'Kassadin': ['Set3_Celestial', 'ManaReaver'],
    'Kayle': ['Valkyrie', 'Set3_Blademaster'],
    'KhaZix': ['Set3_Void', 'Infiltrator'],
    'Leona': ['Cybernetic', 'Vanguard'],
    'Lucian': ['Cybernetic', 'Blaster'],
    'Lulu': ['Set3_Celestial', 'Set3_Mystic'],
    'Lux': ['DarkStar', 'Set3_Sorcerer'],
    'Malphite': ['Rebel', 'Set3_Brawler'],
    'MasterYi': ['Rebel', 'Set3_Blademaster'],
    'MissFortune': ['Valkyrie', 'Blaster', 'Mercenary'],
    'Mordekaiser': ['DarkStar', 'Vanguard'],
    'Neeko': ['StarGuardian', 'Protector'],
    'Poppy': ['StarGuardian', 'Vanguard'],
    'Rakan': ['Set3_Celestial', 'Protector'],
    'Rumble': ['MechPilot', 'Demolitionist'],
    'Shaco': ['DarkStar', 'Infiltrator'],
    'Shen': ['Chrono', 'Set3_Blademaster'],
    'Sona': ['Rebel', 'Set3_Mystic'],
    'Soraka': ['StarGuardian', 'Set3_Mystic'],
    'Syndra': ['StarGuardian', 'Set3_Sorcerer'],
    'Thresh': ['Chrono', 'ManaReaver'],
    'TwistedFate': ['Chrono', 'Set3_Sorcerer'],
    'VelKoz': ['Set3_Void', 'Set3_Sorcerer'],
    'Vi': ['Cybernetic', 'Set3_Brawler'],
    'WuKong': ['Chrono', 'Vanguard'],
    'Xayah': ['Set3_Celestial', 'Set3_Blademaster'],
    'Xerath': ['DarkStar', 'Set3_Sorcerer'],
    'XinZhao': ['Set3_Celestial', 'Protector'],
    'Yasuo': ['Rebel', 'Set3_Blademaster'],
    'Ziggs': ['Rebel', 'Demolitionist'],
    'Zoe': ['StarGuardian', 'Set3_Sorcerer']
}

# 시너지를 주는 아이템 목록
item_to_trait = {
    18: 'Set3_Blademaster',   # Blade of the Ruined King
    28: 'Infiltrator',        # Infiltrator's Talons
    38: 'Demolitionist',      # Demolitionist's Charge
    48: 'StarGuardian',       # Star Guardian's Charm
    58: 'Rebel',              # Rebel Medal
    68: 'Set3_Celestial',     # Celestial Orb
    78: 'Protector',          # Protector's Chestguard
    89: 'DarkStar'            # Dark Star's Heart
}

In [ ]:
# 복구할 데이터를 정리하는 함수 정의
def make_combination_from_champion_and_items(champion_dict):
    # 시너지 카운트 저장용 딕셔너리
    trait_count = {}
    
    # 챔피언 이름, 정보 하나씩 반복
    for champ_name, champ_info in champion_dict.items():
        # 챔피언 기본 시너지 조회
        base_traits = champion_traits.get(champ_name, [])

        # 기본 시너지 개수 누적
        for trait in base_traits:
            trait_count[trait] = trait_count.get(trait, 0) + 1
        
        # 장착 아이템 목록 조회
        item_ids = champ_info.get('items', [])
        
        # 아이템이 추가 시너지를 주는지 확인
        for item_id in item_ids:
            added_trait = item_to_trait.get(item_id)

            # 추가 시너지가 있으면 개수 누적
            if added_trait:
                trait_count[added_trait] = trait_count.get(added_trait, 0) + 1
    
    # 시너지별 개수 반환
    return trait_count

In [ ]:
# 일부 데이터를 복구하기 전 데이터프레임 저장
df_all_match_5.to_csv("./유저단위_게임데이터_상위랭커보존.csv", index=False, encoding='utf-8-sig')
df_all_match_6.to_csv("./유저단위_게임데이터_하위랭커보존.csv", index=False, encoding='utf-8-sig')

### 5-1. df_all_match_5 데이터프레임 데이터 복구
- 동일 게임 ID를 기준으로 여러 티어가 섞여있는 게임이라면, 상위 티어를 가진 유저의 정보만 보존한 데이터프레임
- 티어 믹스매치가 아닌 경기 + 티어 믹스매치 경기 중 상위 랭커의 정보만 유지

In [ ]:
# 비어있는 combination 데이터를 일부 복구하는 함수
def rebuild_combination(row):
    # flag_2가 1인 경우 (combination은 비어있고 champion 데이터가 있는 경우)
    # champion 데이터를 기반으로 combination을 재구성
    
    if row['flag_2'] == 1: # 행 단위로 가져와서 검사하고 실행
        return make_combination_from_champion_and_items(ast.literal_eval(row['champion']))
    
    # flag_2가 0인 경우 (combination 데이터가 정상적으로 있는 경우)
    # 기존 combination 값을 그대로 유지
    else:
        return row['combination']

In [28]:
# 함수를 행 단위로 적용하여 'combination_rebuild' 컬럼의 값을 채울 예정
df_all_match_5['combination_rebuild'] = df_all_match_5.apply(rebuild_combination, axis=1)

In [30]:
# 데이터 복구 결과 확인
df_all_match_5[df_all_match_5['flag_2'] == 1][['combination', 'combination_rebuild']].head()

,combination,combination_rebuild
6627,{},"{'Cybernetic': 2, 'Set3_Blademaster': 1, 'Vanguard': 2, 'Valkyrie': 1, 'Infiltrator': 1, 'DarkStar': 1, 'Set3_Celestial': 1, 'ManaReaver': 1}"
10005,{},"{'DarkStar': 2, 'Protector': 1, 'Set3_Sorcerer': 1}"
15897,{},"{'Rebel': 1, 'Set3_Brawler': 1, 'Cybernetic': 1, 'Vanguard': 2, 'DarkStar': 1, 'Set3_Celestial': 1, 'Sniper': 1}"
20909,{},"{'Cybernetic': 6, 'Set3_Blademaster': 2, 'Vanguard': 2, 'Blaster': 1, 'Set3_Brawler': 1, 'ManaReaver': 2, 'Chrono': 2, 'Infiltrator': 1}"
22014,{},"{'Rebel': 1, 'Set3_Brawler': 1, 'Set3_Void': 1, 'Infiltrator': 2, 'Cybernetic': 1, 'Vanguard': 4, 'DarkStar': 2, 'SpacePirate': 1, 'Chrono': 1}"


### 5-2. df_all_match_6 데이터프레임 데이터 복구
- 동일 게임 ID를 기준으로 여러 티어가 섞여있는 게임이라면, 하위 티어를 가진 유저의 정보만 보존한 데이터프레임
- 티어 믹스매치가 아닌 경기 + 티어 믹스매치 경기 중 하위 랭커의 정보만 유지

In [31]:
# 함수를 행 단위로 적용하여 'combination_rebuild' 컬럼의 값을 채울 예정
df_all_match_6['combination_rebuild'] = df_all_match_6.apply(rebuild_combination, axis=1)

In [32]:
# 데이터 복구 결과 확인
df_all_match_6[df_all_match_6['flag_2'] == 1][['combination', 'combination_rebuild']].head()

,combination,combination_rebuild
6627,{},"{'Cybernetic': 2, 'Set3_Blademaster': 1, 'Vanguard': 2, 'Valkyrie': 1, 'Infiltrator': 1, 'DarkStar': 1, 'Set3_Celestial': 1, 'ManaReaver': 1}"
10005,{},"{'DarkStar': 2, 'Protector': 1, 'Set3_Sorcerer': 1}"
15897,{},"{'Rebel': 1, 'Set3_Brawler': 1, 'Cybernetic': 1, 'Vanguard': 2, 'DarkStar': 1, 'Set3_Celestial': 1, 'Sniper': 1}"
20909,{},"{'Cybernetic': 6, 'Set3_Blademaster': 2, 'Vanguard': 2, 'Blaster': 1, 'Set3_Brawler': 1, 'ManaReaver': 2, 'Chrono': 2, 'Infiltrator': 1}"
22014,{},"{'Rebel': 1, 'Set3_Brawler': 1, 'Set3_Void': 1, 'Infiltrator': 2, 'Cybernetic': 1, 'Vanguard': 4, 'DarkStar': 2, 'SpacePirate': 1, 'Chrono': 1}"


---
## 6. 데이터 복구 결과 확인, csv 파일로 저장

In [33]:
# combination 데이터를 복구한 뒤, csv 파일로 저장
df_all_match_5.to_csv("./유저단위_게임데이터_상위랭커보존-데이터복구완료.csv", index=False, encoding='utf-8-sig')
df_all_match_6.to_csv("./유저단위_게임데이터_하위랭커보존-데이터복구완료.csv", index=False, encoding='utf-8-sig')

In [34]:
# combination_rebuild에 빈 딕셔너리가 있는지 확인
df_all_match_5['combination_rebuild'].apply(
    lambda x: x == {} if isinstance(x, dict) else ast.literal_eval(x) == {}
).sum()

np.int64(0)

In [35]:
# combination_rebuild에 빈 딕셔너리가 있는지 확인
df_all_match_6['combination_rebuild'].apply(
    lambda x: x == {} if isinstance(x, dict) else ast.literal_eval(x) == {}
).sum()

np.int64(0)